# v5.7_v11_5 訓練 — Datasets_YOLO26_v5.7（`arm=base`）

v5.7 的對照臂：只換資料集（`Thrips_Damage` 重標），其餘與 v11.5 逐字相同。

---

## 這一輪在問什麼

兩件事一起驗，而且**互不干擾**（動到的是不同類別）：

| | 改了什麼 | 要看哪一個類別 |
| --- | --- | --- |
| **v5.6 → v5.7** | `Thrips_Damage` 的 208 張依新約定重標（一張葉子一個框，245 框 → 215 框） | `Thrips_Damage` |
| **base → ext** | train 多 124 張外部潛葉蛾影像（配額中性） | `Citrus_Leaf_Miner` |

所以要跑**兩本**：`train_v5_7_base.ipynb` 與 `train_v5_7_ext.ipynb`。
兩本的訓練程式逐字相同，**只有資料集不同**，由 `tools/make_train_notebooks_v5_7.py` 生成。

---

## 判準（先寫死，再看結果）

| # | 問題 | 判準 |
| --- | --- | --- |
| 1 | 外部影像對 CLM 有沒有幫助 | ext 的 `Citrus_Leaf_Miner` AP50 減 base，要**超過 2σ**（平台期 σ ≈ 0.002，逐類雜訊地板 **±0.04**）才算數 |
| 2 | 重標有沒有修好 `Thrips_Damage` 的矛盾 | valid − test 的差距要掉到**顯著門檻 0.165 以內**（v11.5 是 0.202，三輪都沒收斂） |
| 3 | 其餘七類 | **不預期任何變化**；超過 ±0.04 才需要解釋 |

> `Thrips_Damage` 的 **AP 絕對值不能跟 v5.6 比**——框的定義換了，
> 框中位面積從 11.30% 變成 15.40%。能比的是「valid/test 還矛不矛盾」。

---

## 所需資料

| 項目 | 內容 |
| --- | --- |
| 資料集 | **`Datasets_YOLO26_v5.7`**（本機 `Datasets/2_處理與切分/v5.7/`）上傳成 Kaggle Dataset，slug `datasets-yolo26-v5-7` |
| 規模 | 9 類，`train 7,264 / valid 401 / test 401` 影像 |
| 內含 | `data.yaml`、`train/`、`valid/`、`test/`、`_provenance.json` |
| 網路 | **Internet 必須開啟**（要抓 `yolo26n.pt`） |

> Step.2 除了原本的張數指紋，另外**直接讀 `_provenance.json` 的 `meta.version` 與 `meta.arm`**。
> 張數在 base 與 ext 是一樣的（配額中性就是這個意思），光靠張數分不出兩臂——
> 上傳錯資料集會直接停在 Step.2。

---

## 參數

**與 v11.5 逐字相同**（`patience=0`、70 輪、`MuSGD`、`imgsz=640`、`batch=20`、`seed=0`），
一個都沒動。這一輪唯一的變因是資料集。

主要結果一律報 **`last.pt`**：固定輪數跑完的自然終點，valid 沒參與任何決策，
所以 valid 與 test 可以合法併計。`best.pt` 只作對照。

---

## 所需時間

約 **3.4 小時**（訓練 70 輪 3.1 h ＋ 資料集複製與四次評估 0.3 h）。
Kaggle 單場上限 12 h，不需要拆場。兩臂合計約 6.8 h。

建議 **Save & Run All**。


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 唯一需要編輯的 cell
# ══════════════════════════════════════════════════════════════════════
RUN      = "v5.7_v11_5"   # 訓練輸出目錄名稱
EPOCHS   = 70             # ep69 在 v8 / v9-A0 / v11 三次都是最佳輪次
PATIENCE = 0              # 0 = 關閉早停。valid 因此不參與任何決策，可與 test 併計

# Datasets_YOLO26_v5.7（本機 Datasets/2_處理與切分/v5.7）上傳成 Kaggle Dataset 後的路徑
DATASET_SRC = "/kaggle/input/datasets-yolo26-v5-7"
EXPECT_ARM  = "base"        # Step.2 會拿 _provenance.json 對這個值
EXPECT_VER  = "v5.7"

# 本輪不需要拆場：70 輪約 3.1 h，遠低於 Kaggle 的 12 h 上限。
STOP_AFTER_EPOCHS = None
DEADLINE_HOURS    = 10.5

# ── 超參數：與 v11 逐項相同，維持可對照 ────────────────────────────────
HYPERS = dict(
    optimizer="MuSGD", lr0=0.008, lrf=0.01, momentum=0.937, cos_lr=True,
    warmup_epochs=3.0, mosaic=0.7, cls=0.8, dfl=1.5, box=8.0, imgsz=640,
    batch=20, seed=0, cache="ram", workers=4, plots=True, device=0,
)

# close_mosaic 依 epochs 等比推導（v8 是 160 輪關 30 輪），維持排程形狀一致。
# 三次紀錄都顯示它只是讓模型更用力擬合訓練集，但刻意不在這裡動它——
# 要驗證它的去留應該當獨立一臂測，否則本輪與 v11 之間多一個無法歸因的變因。
CLOSE_MOSAIC = max(1, round(30 * EPOCHS / 160))

# v10 實測的 test ±2SE 與當時的 test 影像數，用來投影本輪的 ±2SE（Step.7 會算）
SE_BASELINE = {
    "Oily_Spot": (0.022, 24), "Canker": (0.061, 24), "Sooty_Mold": (0.030, 32),
    "Black_Spot": (0.043, 25), "Scale_Insect": (0.082, 25),
    "Citrus_Leaf_Miner": (0.210, 20), "Thrips": (0.170, 39),
    "Aphid": (0.048, 76), "Thrips_Damage": (0.161, 21),
}

assert PATIENCE == 0, "本輪的全部意義就在 patience=0；要改請先讀第一個 cell"
print(f"▷ RUN={RUN}  epochs={EPOCHS}  patience={PATIENCE}（早停已關閉）"
      f"  close_mosaic={CLOSE_MOSAIC}")


# Step.1 環境

In [ ]:
!nvidia-smi
# 版本釘死：v9 的所有數字都在 8.4.121 上取得，換版本會失去可對照性
!pip install -q ultralytics==8.4.121

import ultralytics
assert ultralytics.__version__ == "8.4.121", \
    f"ultralytics 版本不符：{ultralytics.__version__}"
print(f"▷ ultralytics {ultralytics.__version__}")

# Step.2 資料集準備
### 複製到可寫入工作區、校驗完整性、清 BOM 與舊快取、改寫 data.yaml 路徑。

In [ ]:
import json, os, shutil, sys, yaml

def render_progress_bar(current, total, task_name="檔案同步複製中", bar_length=25):
    percent = (current / total) * 100 if total > 0 else 100.0
    filled = int(bar_length * current // total) if total > 0 else bar_length
    bar = "█" * filled + "░" * (bar_length - filled)
    sys.stdout.write(f"\r▷ 正在執行 [{task_name}] | 進度: [{bar}] {percent:5.1f}% ({current}/{total})")
    sys.stdout.flush()


def copy_and_verify_dataset(src_dir, dst_dir):
    if not os.path.exists(src_dir):
        print(f"▷ 錯誤：找不到來源資料集目錄 {src_dir}")
        return False

    src_files = []
    for root, _, files in os.walk(src_dir):
        for file in files:
            src_files.append(os.path.relpath(os.path.join(root, file), src_dir))
    total_files = len(src_files)
    print(f"▷ 來源資料集掃描完成，共計 {total_files} 個檔案")

    for idx, rel_path in enumerate(src_files, 1):
        dst_path = os.path.join(dst_dir, rel_path)
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        shutil.copy2(os.path.join(src_dir, rel_path), dst_path)
        if idx % 200 == 0 or idx == total_files:
            render_progress_bar(idx, total_files)

    print("\n\n▷ 正在檢查複製檔案")
    dst_files_set = set()
    for root, _, files in os.walk(dst_dir):
        for file in files:
            dst_files_set.add(os.path.relpath(os.path.join(root, file), dst_dir))

    missing, corrupted = [], []
    for rel_path in src_files:
        if rel_path not in dst_files_set:
            missing.append(rel_path)
        elif os.path.getsize(os.path.join(src_dir, rel_path)) != \
                os.path.getsize(os.path.join(dst_dir, rel_path)):
            corrupted.append(rel_path)

    print("≡" * 60)
    print("▷ 資料集複製完整性校驗：")
    print(f"  ▶ 來源檔案總數 : {len(src_files)}")
    print(f"  ▶ 目標檔案總數 : {len(dst_files_set)}")
    print(f"  ▶ 遺漏檔案數   : {len(missing)}")
    print(f"  ▶ 損毀/大小不符: {len(corrupted)}")
    ok = not missing and not corrupted
    print("▷ 檢查通過" if ok else f"▷ 檢查失敗  遺漏={missing[:5]}  損毀={corrupted[:5]}")
    print("≡" * 60)
    return ok


DST = "/kaggle/working/datasets-yolo26-v5-7"
DATA_YAML = "/kaggle/working/data.yaml"

assert copy_and_verify_dataset(DATASET_SRC, DST), "資料集複製失敗，不要往下跑"

# data.yaml 改寫成絕對路徑（來源版本用的是相對的 path: .）
with open(os.path.join(DST, "data.yaml"), encoding="utf-8") as f:
    ycfg = yaml.safe_load(f)
ycfg.update(path=DST, train="train/images", val="valid/images", test="test/images")
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(ycfg, f, default_flow_style=False, allow_unicode=True)

# v5.5 是 9 類。這裡刻意不寫死數字以外的假設：nc 由 data.yaml 決定，
# 後面的模型組態與驗證全部沿用 ycfg["nc"]。
NC = int(ycfg["nc"])
assert NC == 9, f"nc={NC}，應為 9（8 = 舊的 v5r，跑錯資料集了）"
assert ycfg["names"][8] == "Thrips_Damage", f"第 9 類應為 Thrips_Damage，實際是 {ycfg['names'][8]}"

# 這一項是 v11 之後新加的。v5.5 與 v5.6 的 nc 與類別名完全相同，只有評估集大小不同
# （v5.5 是 327/326、v5.6 是 401/401）。v11 有一半的分析時間花在事後確認「到底跑了哪一版」，
# 因為評估腳本裡留了一個寫死的舊字串。拿張數當指紋，跑錯就直接停在這裡。
_n = {sp: len(os.listdir(os.path.join(DST, sp, "images"))) for sp in ("train", "valid", "test")}
print(f"▷ 影像張數 {_n}")
assert _n["valid"] == 401 and _n["test"] == 401, (
    f"valid/test 應為 401/401，實際 {_n['valid']}/{_n['test']}。"
    f"327/326 代表上傳的是 v5.5。")   # v5.6 與 v5.7 同為 401/401，靠下面的 provenance 分辨
assert _n["train"] == 7264, f"train 應為 7,264，實際 {_n['train']}"

# ══════════════════════════════════════════════════════════════════════
# 這一項是 v5.7 新加的，而且比張數重要。
#
# base 與 ext 的張數**完全一樣**（配額中性：外部影像佔用 CLM 既有的增強配額），
# 所以上一段的指紋分不出兩臂——拿錯資料集會一路跑完 3.4 小時才發現。
# _provenance.json 的 meta 直接寫著版本與臂別，拿它當真正的指紋。
# ══════════════════════════════════════════════════════════════════════
with open(os.path.join(DST, "_provenance.json"), encoding="utf-8") as f:
    _meta = json.load(f)["meta"]
print(f"▷ provenance meta: version={_meta.get('version')}  arm={_meta.get('arm')}  "
      f"seed={_meta.get('seed')}  rotate_mode={_meta.get('rotate_mode')}")
assert _meta.get("version") == EXPECT_VER, (
    f"資料集版本是 {_meta.get('version')}，這本 notebook 要的是 {EXPECT_VER}")
assert _meta.get("arm") == EXPECT_ARM, (
    f"資料集臂別是 {_meta.get('arm')}，這本 notebook 要的是 {EXPECT_ARM}。"
    f"base 與 ext 的張數相同，只有這裡分得出來。")

# BOM 與舊快取會讓 Ultralytics 的標註解析出錯
bom_fixed = cache_removed = 0
for subdir, _, files in os.walk(DST):
    for file in files:
        path = os.path.join(subdir, file)
        if file.endswith(".cache"):
            os.remove(path)
            cache_removed += 1
        elif file.endswith(".txt") and "labels" in subdir:
            with open(path, "rb") as fh:
                is_bom = fh.read(3) == b"\xef\xbb\xbf"
            if is_bom:
                with open(path, encoding="utf-8-sig") as fh:
                    content = fh.read()
                with open(path, "w", encoding="utf-8") as fh:
                    fh.write(content)
                bom_fixed += 1

print(f"▷ data.yaml → {DATA_YAML}   nc={NC}")
print(f"▷ names = {ycfg['names']}")
print(f"▷ 修正 BOM {bom_fixed} 個、清除快取 {cache_removed} 個")
print("▷ Step.2 完成")

# Step.3 模型組態
### 直接用 ultralytics 內建的 `yolo26-p2.yaml`，只覆寫 `nc`。不做任何模組替換。

In [ ]:
import yaml as _yaml
from pathlib import Path
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.torch_utils import get_flops, get_num_params

# 讀已安裝的官方組態。v9 時代是先讀進來、再依臂別替換層；v10 一層都不動。
_ULTRA = Path(ultralytics.__file__).parent
cfg = _yaml.safe_load((_ULTRA / "cfg/models/26/yolo26-p2.yaml").read_text(encoding="utf-8"))

cfg["nc"] = NC
cfg["scale"] = "n"
# end2end / reg_max 必須保留：YOLO26 靠這兩個 key 決定走 NMS-free 的 E2EDetectLoss
# （box + cls + l1）還是舊的 DFL(reg_max=16) 路徑。漏掉會靜默變成另一個體系的模型，
# 與 v8 / v9 完全無法對照。官方 yaml 本來就有，這裡只是明示不可被覆寫掉。
cfg.setdefault("end2end", True)
cfg.setdefault("reg_max", 1)

# 檔名必須帶 scale 字母：yaml_model_load 會用檔名覆寫 dict 裡的 scale 鍵，
# 少了 "n" 會落回 scales 字典的第一項並印出 warning。
YAML_PATH = "/kaggle/working/yolo26n-p2-v10.yaml"
with open(YAML_PATH, "w", encoding="utf-8") as f:
    _yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

_m = DetectionModel(cfg=cfg, ch=3, nc=NC, verbose=False)
print(f"▷ v10: {get_num_params(_m):,} params / {get_flops(_m, imgsz=640):.2f} GFLOPs @640")
print(f"▷ yaml → {YAML_PATH}")
del _m

HP = dict(HYPERS)
HP.update(epochs=EPOCHS, patience=PATIENCE, close_mosaic=CLOSE_MOSAIC, name=RUN)
print("▷ 超參數：" + "  ".join(f"{k}={v}" for k, v in sorted(HP.items())))

# Step.4 架構隔離驗證
### 在燒 GPU 時數之前確認：體系正確、可前向、損失有限。

In [ ]:
import math, torch
from ultralytics import YOLO
from ultralytics.cfg import get_cfg
import ultralytics.utils.loss

model = YOLO(YAML_PATH)
core = model.model

# 1. E2E 體系必須與 v8 / v9 一致，否則走的是另一個損失路徑、完全無法對照
det = core.model[-1]
assert getattr(det, "end2end", False), "Detect 不是 end2end：yaml 少了 end2end: True"
assert det.reg_max == 1, f"reg_max={det.reg_max}，應為 1"
assert det.nc == NC and det.nl == 4, f"nc={det.nc} nl={det.nl}"
strides = [int(s) for s in det.stride]
assert strides == [4, 8, 16, 32], f"strides={strides}"
print(f"▷ 1/4 end2end=True, reg_max=1, nc={det.nc}, 偵測頭={det.nl}, strides={strides}")

# 2. 確認是乾淨的官方架構——v10 不該有任何自訂模組或被 patch 過的損失。
#    （這條在 v9 是「該有的模組要就位」，v10 反過來：一個都不該有。）
types = {m.type.split(".")[-1] for m in core.model}
for forbidden in ("ADown", "StarTripletBlock"):
    assert forbidden not in types, f"不該出現 {forbidden}，v10 是原封不動的官方架構"
assert ultralytics.utils.loss.bbox_iou.__name__ == "bbox_iou", \
    f"損失函式被換過了：{ultralytics.utils.loss.bbox_iou.__name__}，v10 應為內建 CIoU"
print("▷ 2/4 無自訂模組、無 loss patch，確認為原封不動的官方 yolo26-p2")

# 3. 前向
core.eval()
with torch.no_grad():
    core(torch.zeros(1, 3, HP["imgsz"], HP["imgsz"]))
print(f"▷ 3/4 Forward pass ({HP['imgsz']}x{HP['imgsz']}) 成功")

# 4. 完整損失路徑，刻意用微小框貼近本資料集分佈
core.args = get_cfg(overrides={"box": HP["box"], "cls": HP["cls"], "dfl": HP["dfl"]})
core.train()
loss, items = core.loss({
    "img": torch.rand(2, 3, HP["imgsz"], HP["imgsz"]),
    "batch_idx": torch.tensor([0.0, 0.0, 1.0]),
    "cls": torch.tensor([[4.0], [8.0], [1.0]]),      # Scale_Insect / Thrips_Damage / Canker
    "bboxes": torch.tensor([[0.50, 0.50, 0.04, 0.04],
                            [0.22, 0.31, 0.20, 0.18],
                            [0.71, 0.68, 0.03, 0.03]]),
})
vals = ({k: float(v) for k, v in items.items()} if isinstance(items, dict)
        else {k: float(v) for k, v in zip(("box", "cls", "l1"), items.flatten())})
assert torch.isfinite(loss).all() and all(math.isfinite(v) for v in vals.values()), vals
print("▷ 4/4 損失路徑通過   " + "  ".join(f"{k}={v:.4f}" for k, v in vals.items()))
print("\n▷ Step.4 全部通過，可以開始訓練")

# Step.5 訓練

In [ ]:
import shutil, time
from ultralytics import YOLO

model = YOLO(YAML_PATH)
model.load("yolo26n.pt")     # v10 架構與官方一致，轉移率應為 902/902


def stop_and_snapshot(trainer):
    """每輪保留可續跑的 checkpoint，並在時數/輪數上限時乾淨停止。

    訓練迴圈結束後一定會執行 final_eval() → strip_optimizer()，把 last.pt / best.pt
    的 epoch 改成 -1 並清掉 optimizer/EMA，那種檔案無法續跑。
    on_fit_epoch_end 的觸發點在 save_model() 之後、跳出迴圈之前，此時 last.pt
    才剛寫好且尚未被 strip。

    備份不設條件：patience 早停時 trainer.stop 在進入這個 callback 之前就已為 True，
    若寫成 `if not trainer.stop` 這段會整個被跳過——那正是 v9 踩過的失效模式。
    本輪 patience=0 不會早停，但這段照留：它同時也是牆鐘超時的退場路徑。
    """
    if trainer.last.exists():
        shutil.copy(trainer.last, trainer.wdir / "resume_from.pt")

    cap = STOP_AFTER_EPOCHS or trainer.epochs      # None → 跑滿，不提前停
    elapsed = (time.time() - trainer.train_time_start) / 3600
    if not trainer.stop and (trainer.epoch + 1 >= cap or elapsed > DEADLINE_HOURS):
        trainer.stop = True
        print(f"\n▷ 停於第 {trainer.epoch + 1} / {trainer.epochs} 輪，已耗時 {elapsed:.2f} h")
        print(f"▷ 續跑用 checkpoint：{trainer.wdir / 'resume_from.pt'}")


model.add_callback("on_fit_epoch_end", stop_and_snapshot)

_cap = STOP_AFTER_EPOCHS or EPOCHS
print(f"▷ 將跑到第 {_cap} / {EPOCHS} 輪"
      + ("" if _cap >= EPOCHS else "  ← STOP_AFTER_EPOCHS 會提前停止，剩餘輪次需用 RESUME.ipynb")
      + f"；牆鐘上限 {DEADLINE_HOURS} h")

results = model.train(data=DATA_YAML, save_period=10, **HP)
print("▷ v11.5 訓練完畢")

# Step.6 輸出整理

In [ ]:
import os, shutil

runs_dir = "/kaggle/working/runs"
if os.path.exists(runs_dir):
    print("▷ 正在壓縮訓練輸出")
    shutil.make_archive(f"/kaggle/working/runs_{RUN}", "zip", runs_dir)
    size = os.path.getsize(f"/kaggle/working/runs_{RUN}.zip") / (1024 ** 2)
    print(f"▷ 壓縮成功 /kaggle/working/runs_{RUN}.zip ({size:.2f} MB)")
else:
    print(f"▷ 壓縮失敗：找不到 {runs_dir}")

# Step.7 評估包

2 個權重 × 2 個 split 共四次評估。**`last.pt` 是主要結果**——固定輪數跑完的終點，valid 沒有參與任何決策，所以 valid 與 test 可以合法併計；`best.pt` 只作對照，用來量 best-of-N 灌水有多大。`summary.json` 直接寫出 pooled 的逐類 ±2SE。

In [ ]:
import csv, json, os, shutil, zipfile
from ultralytics import YOLO


def write_csv(path, fieldnames, rows):
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)


RUN_DIR = str(model.trainer.save_dir)
OUT = f"/kaggle/working/eval_{RUN}"
os.makedirs(OUT, exist_ok=True)

# ══════════════════════════════════════════════════════════════════════
# 2 個權重 × 2 個 split，共四次評估
#
#   last.pt  ep70 固定輪數跑完的自然終點，valid 完全沒參與任何決策
#            → **valid 與 test 可以合法併計**，這是本輪的主要結果
#   best.pt  由 fitness（在 valid 上算）挑出，帶選擇偏誤
#            → 只作對照，用來量 best-of-N 灌水有多大
#
# plots=True 是必要的，不是為了畫圖：ultralytics 把 confusion_matrix.process_batch
# 包在 `if self.args.plots` 裡（detect/val.py:196），plots=False 會讓混淆矩陣維持全零。
# ══════════════════════════════════════════════════════════════════════
WEIGHTS = {"last": os.path.join(RUN_DIR, "weights", "last.pt"),
           "best": os.path.join(RUN_DIR, "weights", "best.pt")}
evals = {}

for tag, wp in WEIGHTS.items():
    for split in ("val", "test"):
        mm = YOLO(wp).val(data=DATA_YAML, split=split, imgsz=HP["imgsz"],
                          batch=HP["batch"], plots=True)
        nm = mm.names if isinstance(mm.names, dict) else dict(enumerate(mm.names))
        per = {}
        for i, ci in enumerate(mm.box.ap_class_index):
            ci = int(ci)
            per[nm.get(ci, str(ci))] = {
                "precision": round(float(mm.box.p[i]), 5),
                "recall": round(float(mm.box.r[i]), 5),
                "f1": round(float(mm.box.f1[i]), 5),
                "ap50": round(float(mm.box.ap50[i]), 5),
                "ap50_95": round(float(mm.box.ap[i]), 5),
            }
        evals[f"{tag}_{split}"] = {
            "mAP50": round(float(mm.box.map50), 5),
            "mAP50_95": round(float(mm.box.map), 5),
            "precision": round(float(mm.box.mp), 5),
            "recall": round(float(mm.box.mr), 5),
            "per_class": per,
        }
        print(f"▷ {tag}.pt @ {split}   mAP50 {mm.box.map50:.5f}   mAP50-95 {mm.box.map:.5f}")
        if tag == "last" and split == "test":
            m_main, cm_main = mm, mm.confusion_matrix.matrix
            names = nm

# ── 逐輪指標與超參數 ────────────────────────────────────────────────
for fn in ("results.csv", "args.yaml"):
    src = os.path.join(RUN_DIR, fn)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(OUT, fn))

# ── 每類指標（四種組合各一份）──────────────────────────────────────
COLS = ["class", "precision", "recall", "f1", "ap50", "ap50_95"]
for k, e in evals.items():
    write_csv(os.path.join(OUT, f"per_class_{k}.csv"), COLS,
              [{"class": c, **v} for c, v in e["per_class"].items()])

# ── 混淆矩陣（last.pt @ test；列=預測，欄=真實，最後一列/欄為背景）──
nc = len(names)
labels = [names.get(i, str(i)) for i in range(nc)] + ["background"]
with open(os.path.join(OUT, "confusion_matrix.csv"), "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow([""] + [f"true_{l}" for l in labels])
    for i, lab in enumerate(labels):
        w.writerow([f"pred_{lab}"] + [int(cm_main[i][j]) for j in range(len(labels))])
fp_bg = {labels[i]: int(cm_main[i][nc]) for i in range(nc)}
fn_bg = {labels[i]: int(cm_main[nc][i]) for i in range(nc)}

# ── F1-信心曲線與最佳截斷點（last.pt @ test）────────────────────────
best_conf = None
try:
    x, y, _xl, _yl = m_main.curves_results[1]        # F1-Confidence(B)
    x = [float(v) for v in x]
    mean_f1 = ([sum(col) / len(col) for col in zip(*y)] if hasattr(y[0], "__len__")
               else [float(v) for v in y])
    write_csv(os.path.join(OUT, "f1_conf.csv"), ["conf", "mean_f1"],
              [{"conf": a, "mean_f1": b} for a, b in zip(x, mean_f1)])
    best_conf = round(x[mean_f1.index(max(mean_f1))], 4)
except Exception as e:
    print(f"▷ F1-conf 曲線取用失敗（不影響主要結果）：{type(e).__name__}: {e}")

# ── 平台期統計 ──────────────────────────────────────────────────────
plateau = {}
rcsv = os.path.join(OUT, "results.csv")
if os.path.exists(rcsv):
    with open(rcsv, encoding="utf-8") as f:
        rec = [{k.strip(): v for k, v in row.items()} for row in csv.DictReader(f)]
    win = 50 if EPOCHS >= 120 else 16
    tail = rec[-win:]
    for key, col in (("mAP50", "metrics/mAP50(B)"), ("mAP50_95", "metrics/mAP50-95(B)")):
        if rec and col in rec[0]:
            vals = [float(r[col]) for r in tail]
            mean = sum(vals) / len(vals)
            var = sum((v - mean) ** 2 for v in vals) / max(len(vals) - 1, 1)
            plateau[key] = {"window": win, "mean": round(mean, 5),
                            "std": round(var ** 0.5, 5), "best": round(max(vals), 5)}

# ══════════════════════════════════════════════════════════════════════
# 本輪的重點：pooled（valid + test）的逐類 ±2SE
#
# 只有 last.pt 能這樣算——它是固定輪數跑完的終點，valid 沒有參與任何決策。
# ±2SE 由 v10 實測值依 1/sqrt(n) 投影；真值要事後跑 diag_localization.py --bootstrap。
# ══════════════════════════════════════════════════════════════════════
N_EVAL = {"Oily_Spot": (24, 24), "Canker": (24, 24), "Sooty_Mold": (33, 32),
          "Black_Spot": (25, 25), "Scale_Insect": (35, 35),
          "Citrus_Leaf_Miner": (45, 45), "Thrips": (60, 60),
          "Aphid": (75, 76), "Thrips_Damage": (40, 40)}

pooled, worst = {}, 0.0
print(f"\n{'類別':<20}{'valid AP50':>11}{'test AP50':>10}{'差':>8}{'pooled ±2SE':>13}{'':>4}")
for c, (nv, nt) in N_EVAL.items():
    va = evals["last_val"]["per_class"].get(c, {}).get("ap50")
    ta = evals["last_test"]["per_class"].get(c, {}).get("ap50")
    se0, n0 = SE_BASELINE[c]
    se = round(se0 * (n0 / (nv + nt)) ** 0.5, 4)
    worst = max(worst, se)
    gap = None if (va is None or ta is None) else round(ta - va, 4)
    pooled[c] = {"n_valid": nv, "n_test": nt, "ap50_valid": va, "ap50_test": ta,
                 "gap": gap, "se2_pooled": se, "meets_target": se <= 0.10}
    print(f"{c:<20}{(va if va is not None else float('nan')):>11.3f}"
          f"{(ta if ta is not None else float('nan')):>10.3f}"
          f"{(gap if gap is not None else float('nan')):>8.3f}"
          f"{se:>13.3f}{'✓' if se <= 0.10 else '✗':>4}")
print(f"\n▷ 最差 pooled ±2SE = {worst:.3f}   目標 0.10   "
      f"{'★ 達成' if worst <= 0.10 else '✗ 未達成'}")

inflation = None
if plateau.get("mAP50_95"):
    inflation = round(evals["best_val"]["mAP50_95"] - plateau["mAP50_95"]["mean"], 5)
    sd = plateau["mAP50_95"]["std"] or 1e-9
    print(f"▷ best.pt 高出平台期 {inflation:+.5f} = {inflation / sd:.1f}σ"
          f"（v10 是 2.8σ、v11 是 2.7σ）")

summary = {
    "run": RUN, "epochs": EPOCHS, "patience": PATIENCE, "close_mosaic": CLOSE_MOSAIC,
    "dataset": _meta.get("version", EXPECT_VER),   # 來自 _provenance.json，不是寫死的
    "arm": _meta.get("arm", EXPECT_ARM),
    "nc": NC, "imgsz": HP["imgsz"],
    "primary_weight": "last.pt",
    "note": ("patience=0 且固定輪數，valid 未參與任何決策，"
             "因此 last.pt 的 valid 與 test 可合法併計；best.pt 僅作對照。"),
    "evals": evals,
    "pooled_2se": pooled,
    "worst_pooled_2se": round(worst, 4),
    "plateau": plateau,
    "best_vs_plateau": inflation,
    "best_f1_conf": best_conf,
    "fp_from_background": fp_bg,
    "fn_to_background": fn_bg,
}
with open(os.path.join(OUT, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

shutil.make_archive(f"/kaggle/working/eval_{RUN}", "zip", OUT)
print(f"\n▷ 評估包 → /kaggle/working/eval_{RUN}.zip")
print("▷ 主要結果（last.pt）：")
print(json.dumps({k: evals[k] for k in ("last_val", "last_test")}, ensure_ascii=False,
                 indent=2, default=str)[:600] + " …")
